In [1]:
!pip install wbgapi pandas plotly kaleido

import wbgapi as wb
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import statsmodels.api as sm

# Same European countries
europe = ['DEU','FRA','ITA','ESP','POL','NLD','BEL','AUT',
          'SWE','CZE','HUN','PRT','GRC','ROU','SVK','FIN']

# CBAM sectors - CO2 emissions intensity (kg per kg of output)
# Source: European Environment Agency / CBAM documentation
cbam_risk = {
    'Steel & Metals': 0.85,
    'Cement': 0.90,
    'Aluminum': 0.75,
    'Fertilizers': 0.70,
    'Electricity': 0.65
}

# Pull manufacturing value added (our proxy for industrial exposure)
mfg = wb.data.DataFrame(
    'NV.IND.MANF.ZS',
    economy=europe,
    time=range(2019, 2023)
)

mfg = mfg.T
mfg.index = mfg.index.str.replace('YR','').astype(int)

print("Data loaded!")
print(mfg.tail())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 kB 1.6 MB/s eta 0:00:00
Data loaded!
economy        AUT        BEL        CZE        DEU        ESP        FIN  \
2019     17.102286  12.413251  22.126677  19.309469  10.696732  14.712097   
2020     16.597326  12.012009  20.796823  18.527496  10.853213  14.523218   
2021     16.979449  10.744808  19.640982  18.572620  11.256085  14.940265   
2022     16.651579  12.533005  19.395061  18.321774  11.027988  15.755162   

economy       FRA       GRC        HUN        ITA        NLD        POL  \
2019     9.910950  7.839590  17.326162  14.803282  10.420757  16.661488   
2020     9.245667  8.427280  17.273991  14.477256  10.484002  16.208369   
2021     9.095825  8.665590  16.527043  15.365301  10.754977  17.300616   
2022     9.301362  9.721029  16.790459  15.382254  10.347548  17.651321   

economy        PRT        ROU        SVK        SWE  
2019     11.937

In [7]:
import pandas as pd

# Direct CSV from Our World in Data - reliable CO2 data
url = "https://raw.githubusercontent.com/owid/co2-data/master/owid-co2-data.csv"
co2_raw = pd.read_csv(url)

# Filter Europe + recent year
europe_names = {
    'DEU': 'Germany', 'FRA': 'France', 'ITA': 'Italy',
    'ESP': 'Spain', 'POL': 'Poland', 'NLD': 'Netherlands',
    'BEL': 'Belgium', 'AUT': 'Austria', 'SWE': 'Sweden',
    'CZE': 'Czechia', 'HUN': 'Hungary', 'PRT': 'Portugal',
    'GRC': 'Greece', 'ROU': 'Romania', 'SVK': 'Slovakia',
    'FIN': 'Finland'
}

co2_filtered = co2_raw[
    (co2_raw['country'].isin(europe_names.values())) &
    (co2_raw['year'] == 2019)
][['country', 'co2_per_capita']].dropna()

# Add country codes
name_to_code = {v: k for k, v in europe_names.items()}
co2_filtered['code'] = co2_filtered['country'].map(name_to_code)

print(co2_filtered.sort_values('co2_per_capita', ascending=False))

           country  co2_per_capita code
12226      Czechia           9.569  CZE
32231  Netherlands           8.709  NLD
5434       Belgium           8.685  BEL
18314      Germany           8.481  DEU
37605       Poland           8.304  POL
17161      Finland           7.687  FIN
3880       Austria           7.651  AUT
41708     Slovakia           6.211  SVK
18764       Greece           6.133  GRC
23357        Italy           5.648  ITA
43403        Spain           5.285  ESP
21307      Hungary           5.031  HUN
17378       France           4.812  FRA
37780     Portugal           4.600  PRT
44115       Sweden           3.967  SWE
38130      Romania           3.933  ROU


In [8]:
# Merge CO2 with manufacturing data
co2_indexed = co2_filtered.set_index('code')['co2_per_capita']
mfg_2020 = mfg.loc[2020].dropna()

cbam_df = pd.DataFrame({
    'co2_per_capita': co2_indexed,
    'manufacturing_share': mfg_2020
}).dropna()

# CBAM Exposure Index
cbam_df['cbam_exposure'] = (
    cbam_df['manufacturing_share'] *
    cbam_df['co2_per_capita'] / 100
)

cbam_df = cbam_df.sort_values('cbam_exposure', ascending=False)
cbam_df['country_code'] = cbam_df.index

print(cbam_df.round(2))

# Map
fig_map = px.choropleth(
    cbam_df,
    locations='country_code',
    color='cbam_exposure',
    color_continuous_scale='RdYlGn_r',
    scope='europe',
    title='CBAM Climate Shock Exposure Index — Europe (2019–2020)'
)
fig_map.show()

fig_map.write_html("climate_shock_map.html", include_plotlyjs='cdn')
from google.colab import files
files.download("climate_shock_map.html")

     co2_per_capita  manufacturing_share  cbam_exposure country_code
CZE            9.57                20.80           1.99          CZE
DEU            8.48                18.53           1.57          DEU
POL            8.30                16.21           1.35          POL
AUT            7.65                16.60           1.27          AUT
FIN            7.69                14.52           1.12          FIN
SVK            6.21                17.54           1.09          SVK
BEL            8.68                12.01           1.04          BEL
NLD            8.71                10.48           0.91          NLD
HUN            5.03                17.27           0.87          HUN
ITA            5.65                14.48           0.82          ITA
ROU            3.93                15.92           0.63          ROU
ESP            5.28                10.85           0.57          ESP
PRT            4.60                12.09           0.56          PRT
GRC            6.13               

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.




<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
import kaleido
kaleido.write_fig(fig_map, "climate_shock_map.html")
from google.colab import files
files.download("climate_shock_map.html")

/tmp/ipykernel_6142/3245715044.py:2: RuntimeWarning:

coroutine 'write_fig' was never awaited



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
# Does CBAM exposure predict unemployment?
unemp = wb.data.DataFrame(
    'SL.UEM.TOTL.ZS',
    economy=europe,
    time=range(2020, 2021)
)
# Corrected: Use the entire column of unemployment data for 2020
unemp_2020 = unemp['SL.UEM.TOTL.ZS'].dropna()

merged = cbam_df.copy()
merged['unemployment'] = merged.index.map(unemp_2020)
merged = merged.dropna()

X = sm.add_constant(merged['cbam_exposure'])
y = merged['unemployment']
model = sm.OLS(y, X).fit()
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:           unemployment   R-squared:                       0.394
Model:                            OLS   Adj. R-squared:                  0.350
Method:                 Least Squares   F-statistic:                     9.089
Date:                Thu, 04 Jun 2026   Prob (F-statistic):            0.00928
Time:                        08:03:56   Log-Likelihood:                -40.072
No. Observations:                  16   AIC:                             84.14
Df Residuals:                      14   BIC:                             85.69
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const            12.3638      1.953      6.330